# Statistical Business Analysis
## Project: Marketing and Sales Performance Analysis

This notebook conducts a comprehensive statistical analysis of business data, including descriptive statistics, hypothesis testing, correlation analysis, and regression.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.api as sm
from statsmodels.formula.api import ols

# Set visual style
sns.set(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (10, 6)

# Load data
df = pd.read_csv('business_data.csv')
df['Date'] = pd.to_datetime(df['Date'])
df.head()

## 1. Descriptive Statistics
Generating mean, median, mode, and standard deviation for key metrics.

In [ ]:
metrics = ['Marketing_Spend', 'Sales_Revenue', 'Customer_Count']
stats_summary = df[metrics].describe().T
stats_summary['mode'] = df[metrics].mode().iloc[0]
stats_summary = stats_summary[['mean', '50%', 'mode', 'std', 'min', 'max']]
stats_summary.columns = ['Mean', 'Median', 'Mode', 'Std Dev', 'Min', 'Max']
print("--- Descriptive Statistics ---")
print(stats_summary)

## 2. Data Distribution Analysis
Visualizing distributions and testing for normality.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.histplot(df['Sales_Revenue'], kde=True, ax=axes[0])
axes[0].set_title('Sales Revenue Distribution')

sns.histplot(df['Marketing_Spend'], kde=True, ax=axes[1])
axes[1].set_title('Marketing Spend Distribution')

plt.show()

# Shapiro-Wilk Test for Normality
_, p_val_sales = stats.shapiro(df['Sales_Revenue'])
print(f"Shapiro-Wilk Test p-value (Sales): {p_val_sales:.4f}")
if p_val_sales > 0.05:
    print("Sales Revenue appears to be normally distributed (p > 0.05)")
else:
    print("Sales Revenue is not normally distributed (p < 0.05)")

## 3. Correlation Analysis
Analyzing the relationship between Marketing Spend and Sales Revenue.

In [ ]:
corr_matrix = df[metrics].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()

pearson_corr, p_corr = stats.pearsonr(df['Marketing_Spend'], df['Sales_Revenue'])
print(f"Pearson Correlation (Spend vs Sales): {pearson_corr:.4f} (p-value: {p_corr:.4f})")

## 4. Hypothesis Testing
### Test 1: Independent T-Test (Sales in East vs West Region)
**H0:** There is no significant difference in sales between East and West regions.
**H1:** There is a significant difference.

In [ ]:
east_sales = df[df['Region'] == 'East']['Sales_Revenue']
west_sales = df[df['Region'] == 'West']['Sales_Revenue']

t_stat, p_val_t = stats.ttest_ind(east_sales, west_sales)
print(f"T-Test (East vs West) p-value: {p_val_t:.4f}")
if p_val_t < 0.05:
    print("Result: Statistically Significant difference found.")
else:
    print("Result: No significant difference found.")

### Test 2: ANOVA (Sales across Marketing Channels)
**H0:** Mean sales are equal across all marketing channels.
**H1:** At least one channel mean is different.

In [ ]:
model_anova = ols('Sales_Revenue ~ C(Marketing_Channel)', data=df).fit()
anova_table = sm.stats.anova_lm(model_anova, typ=2)
print(anova_table)

if anova_table['PR(>F)'][0] < 0.05:
    print("ANOVA Result: Significant difference across channels.")
else:
    print("ANOVA Result: No significant difference across channels.")

## 5. Confidence Intervals
Calculating the 95% Confidence Interval for Average Sales Revenue.

In [ ]:
mean_sales = df['Sales_Revenue'].mean()
sem_sales = stats.sem(df['Sales_Revenue'])
confidence = 0.95
ci = stats.t.interval(confidence, len(df)-1, loc=mean_sales, scale=sem_sales)

margin_of_error = ci[1] - mean_sales
print(f"Mean Sales: ${mean_sales:.2f}")
print(f"95% Confidence Interval: (${ci[0]:.2f}, ${ci[1]:.2f})")
print(f"Margin of Error: ±${margin_of_error:.2f}")

## 6. Regression Analysis
Linear regression to predict Sales Revenue based on Marketing Spend.

In [ ]:
X = df['Marketing_Spend']
y = df['Sales_Revenue']
X = sm.add_constant(X) # Add intercept

model_reg = sm.OLS(y, X).fit()
print(model_reg.summary())

plt.scatter(df['Marketing_Spend'], df['Sales_Revenue'], alpha=0.5)
plt.plot(df['Marketing_Spend'], model_reg.predict(X), color='red')
plt.xlabel('Marketing Spend')
plt.ylabel('Sales Revenue')
plt.title('Linear Regression: Spend vs Revenue')
plt.show()